# 04 · Input encoding and shared representation (paper §2.2)

**Encoding** - the paper encodes numbers *without* discretisation (the token's embedding is
scaled by the standardised value). Compared on METABRIC and SUPPORT with the same model:
`continuous` (the paper), `quantile` (rank-normalised values, same embedding) and
`discretised` (each number cut into bins, one token per bin).

**Representation** - how the transformer's hidden states become one patient vector:
token pooling over layers (sum / mean / concatenate / BERT [CLS]) x pooling over tokens
(max / mean) x layers (all / last).

## Options

In [ ]:
NAME = "04_ablation_encoding_representation"

SEEDS = list(range(10))   # protocol: 10 random 60/10/30 splits (as SurvTRACE); seed s = split s and init s

DATASETS = ["metabric", "support"]
ENCODINGS = ["continuous", "quantile", "discretised"]
# (token_emb, sentence_emb, layers): token_emb 2=sum 3=mean 4=concat 5=[CLS];
# sentence_emb 2=max 3=mean; layers "all" or "last"
REPRESENTATIONS = [(t, s, l) for t in (2, 3, 4) for s in (2, 3) for l in ("all", "last")] + [(5, 1, "all")]

# --- execution ------------------------------------------------------------------------
SMOKE_TEST   = False    # True: 2 epochs, 1 seed, 1-2 configurations - only to check that it runs
SMOKE_EPOCHS = 2
WORKERS      = None     # parallel runs; None = 2 on Kaggle (4 vCPU), 4 locally
TIME_BUDGET_MIN = 11 * 60   # stop launching new runs after this (Kaggle sessions end at 12 h);
                            # re-run with this notebook's output attached to continue
INSTALL_DEPS = True     # Kaggle only
GIT_URL      = "https://github.com/Parsagh05/dmmst.git"   # the code is cloned from here
CLEAN_UP_AT_END = True  # Kaggle: delete the working repo copy after the report

## Setup (identical in every notebook)

In [ ]:
# ---- 1. get the code --------------------------------------------------------------
import os, sys, json, shutil, subprocess, time, hashlib
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").is_dir()

def _is_repo(p):
    return (p / "sat" / "finetune.py").is_file() and (p / "conf").is_dir()

if ON_KAGGLE:
    WORK = Path("/kaggle/working")
    REPO = WORK / "dmmst"
    if not _is_repo(REPO):
        subprocess.run(["git", "clone", "--depth", "1", GIT_URL, str(REPO)], check=True)
    RESULTS_ROOT = WORK / "results"
    INPUT_ROOTS = [Path("/kaggle/input")]
else:
    REPO = Path.cwd().resolve()
    while not _is_repo(REPO) and REPO != REPO.parent:
        REPO = REPO.parent
    RESULTS_ROOT = REPO / "results" / "version_2"
    INPUT_ROOTS = []
assert _is_repo(REPO), REPO
os.chdir(REPO)
sys.path.insert(0, str(REPO))
CODE_FINGERPRINT = hashlib.md5(b"".join(
    p.read_bytes() for p in sorted((REPO / "sat").rglob("*.py")) + sorted((REPO / "conf").rglob("*.yaml"))
)).hexdigest()[:10]
COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO, text=True,
                        capture_output=True).stdout.strip()
print("repo:", REPO, "| commit:", COMMIT, "| code fingerprint:", CODE_FINGERPRINT)

In [ ]:
# ---- 2. dependencies (Kaggle) ------------------------------------------------------
# numpy and torch are NEVER moved: pandas/pyarrow and CUDA in the image are built for them.
if ON_KAGGLE and INSTALL_DEPS:
    import numpy as _np, torch
    _tv = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
    pkgs = ["transformers==4.50.0", "datasets==3.4.1", "tokenizers>=0.21,<0.22", "evaluate>=0.4.3",
            "accelerate>=1.4.0", "hydra-core>=1.3.2", "hydra-colorlog>=1.2.0",
            "torchsurv" if _tv >= (2, 8) else "torchsurv==0.1.4",
            "h5py>=3.13", "logdecorator>=2.5", "einops==0.8.1", "torchtuples>=0.2.2",
            "numba", "nvidia-ml-py", "polars", "tqdm", "tabulate",
            "scikit-survival" if int(_np.__version__.split(".")[0]) >= 2 else "scikit-survival==0.22.2",
            "pycox==0.3.0", "peft", "easydict", "lifelines", "osqp", "patsy",
            f"numpy=={_np.__version__}", f"torch=={torch.__version__.split('+')[0]}"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
    # SurvivalEVAL declares torch>=2.13; it runs on older torch, so its pins are skipped
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "SurvivalEVAL==0.8.5"], check=True)
import torch
print("torch", torch.__version__, "| cuda" if torch.cuda.is_available() else "| cpu")

In [ ]:
# ---- 3. runner ----------------------------------------------------------------------
from IPython.display import Markdown, display
import pandas as pd
from scripts.runner import Runner, select_best
from scripts import v2

RUN_NAME = NAME + ("_smoke" if SMOKE_TEST else "")
r = Runner(repo=REPO, results_dir=RESULTS_ROOT / RUN_NAME,
           workers=WORKERS or (2 if ON_KAGGLE else 4),
           smoke_epochs=SMOKE_EPOCHS if SMOKE_TEST else None, gpu="0")

# Resume: attach this notebook's previous output ("Add Input") and finished runs are
# copied back, so only the missing ones are trained.
n_copied = 0
for root in INPUT_ROOTS:
    for d in root.rglob(RUN_NAME):
        if d.is_dir() and d != r.results_dir:
            for f in d.glob("*.json"):
                if not (r.results_dir / f.name).exists():
                    shutil.copy2(f, r.results_dir / f.name); n_copied += 1
print(f"resumed {n_copied} files from attached inputs")
(r.results_dir / "environment.json").write_text(json.dumps({
    "commit": COMMIT, "code_fingerprint": CODE_FINGERPRINT, "torch": torch.__version__,
    "cuda": torch.cuda.is_available(), "smoke": SMOKE_TEST, "name": RUN_NAME}, indent=2))
print("results ->", r.results_dir)

In [ ]:
# ---- settings tuned on validation in notebook 01 -------------------------------------
TUNED = v2.load_tuned(v2.find_tuned(RESULTS_ROOT, INPUT_ROOTS))
LOSS = TUNED["losses"]
print(json.dumps(TUNED, indent=1)[:1500])

In [ ]:
def tuned(model, name):
    return v2.tuned_overrides(TUNED, model, name)

def loss_extra(name):
    """Per-dataset extras: time unit chosen in notebook 01."""
    return ["v2_time_scale=1.0"] if LOSS[name].get("time_scale_raw") else []

def surv_recipes(name):
    L = LOSS[name]
    multi = v2.DATASETS[name]["events"] > 1
    return v2.survival_recipes(multi, L["rank"], L.get("mul", {"coeff": 0, "sigma": 0.1}))

def reg_recipes(name):
    L = LOSS[name]
    return v2.regression_recipes(v2.DATASETS[name]["events"] > 1, L.get("mm_coeff", 0.5))

GROUP = {"metabric": "survtrace_metabric", "support": "survtrace_support"}
def enc_overrides(ds, enc):
    """(overrides, model-hub folder) of one input encoding."""
    if enc == "continuous":
        return [f"dataset={ds}_numeric"], f"{ds}_numeric"
    if enc == "quantile":
        return [f"dataset={ds}_quantile", "data.parse.scale_method=quantile"], f"{ds}_quantile"
    return [f"data/load={ds}", f"data/parse={ds}", f"tokenizers={ds}", f"dataset={ds}"], ds

## Encodings

In [ ]:
from scripts.runner import Run
if SMOKE_TEST:
    SEEDS, REPRESENTATIONS = SEEDS[:1], REPRESENTATIONS[:2]
runs = []
for ds in DATASETS:
    for enc in ENCODINGS:
        ov, hub = enc_overrides(ds, enc)
        r.prepare(f"{GROUP[ds]}/survival", ov, label=f"{ds}_{enc}", split_seeds=SEEDS)
        for s in SEEDS:
            runs.append(Run(f"{ds}__enc_{enc}__s{s}", "finetune", f"{GROUP[ds]}/survival",
                            ov + v2.V2 + ["tasks=v2_survival"] + tuned("ours", ds) + [f"seed={s}", f"split_seed={s}"],
                            hub, {"dataset": ds, "model": "ours", "variant": f"enc_{enc}", "seed": s, "stage": "encoding"}))
print(len(runs), "encoding runs")
r.run_many(runs, time_budget_min=TIME_BUDGET_MIN)

## Representations

In [ ]:
runs = []
v2.prepare(r, DATASETS, SEEDS, REPO)
for ds in DATASETS:
    for t, s_emb, layers in REPRESENTATIONS:
        extra = tuned("ours", ds) + [f"token_emb={t}", f"sentence_emb={s_emb}"]
        if layers == "last":
            extra.append("select_hidden_layers=[-1]")
        runs += v2.runs("ours", ds, SEEDS, REPO, extra=extra, tag=f"rep_t{t}_s{s_emb}_{layers}",
                        info={"stage": "representation", "token_emb": t, "sentence_emb": s_emb, "layers": layers})
print(len(runs), "representation runs")
r.run_many(runs, time_budget_min=TIME_BUDGET_MIN)

## Tables

In [ ]:
# ---- tables + zip ---------------------------------------------------------------------
rep = r.results_dir / "report"
rep.mkdir(exist_ok=True)
res = r.collect()
res.to_csv(rep / "all_runs.csv", index=False)
if len(res):
    t = v2.table(res, ["dataset", "model", "variant"])
    t.to_csv(rep / "summary.csv", index=False)
    display(t)
print("zip:", r.zip())

## Clean up

In [ ]:
if ON_KAGGLE and CLEAN_UP_AT_END:
    shutil.rmtree(REPO, ignore_errors=True)
    print("removed", REPO, "- results are in", r.results_dir)